In [4]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table

In [5]:
conn = connect_to_postgres()
if conn:
    df = read_table("SELECT * FROM silver.categories LIMIT 100", conn)
    conn.close()
    

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [6]:
df.head()

,snapshot_date,supermarket,category_lvl1_name,category_lvl2_name,category_lvl3_name,category_lvl1_id,category_lvl2_id,category_lvl3_id,category_lvl1_slug,category_lvl2_slug,category_lvl3_slug,created_at
0,2025-08-10,biggie,Alimentos Especiales,None,None,6,None,None,alimentos-especiales,None,None,2025-10-26 21:55:31.862512
1,2025-08-10,biggie,Almacén,None,None,1,None,None,almacen,None,None,2025-10-26 21:55:31.862512
2,2025-08-10,biggie,Asado,None,None,246,None,None,asado,None,None,2025-10-26 21:55:31.862512
3,2025-08-10,biggie,Bebes,None,None,41,None,None,bebes,None,None,2025-10-26 21:55:31.862512
4,2025-08-10,biggie,Bebidas con Alcohol,None,None,3,None,None,bebidas-con-alcohol,None,None,2025-10-26 21:55:31.862512


### Price problem

In [7]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )
'''
print(query)


SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )



In [21]:
conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45260 entries, 0 to 45259
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   snapshot_date    45260 non-null  object        
 1   supermarket      45260 non-null  object        
 2   product_id       16880 non-null  object        
 3   product_name     45260 non-null  object        
 4   brand            15990 non-null  object        
 5   price            45260 non-null  object        
 6   unit_of_measure  23476 non-null  object        
 7   is_on_promotion  6770 non-null   object        
 8   promotion_price  6770 non-null   object        
 9   category_slug    45260 non-null  object        
 10  ingestion_time   45260 non-null  datetime64[ns]
 11  created_at       45260 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(10)
memory usage: 4.1+ MB


In [23]:
df['snapshot_date'].unique()

array([datetime.date(2025, 10, 26)], dtype=object)

In [24]:
df['supermarket'].unique()

array(['biggie', 'real', 'stock', 'casarica'], dtype=object)

In [29]:
df_test = df.loc[df['supermarket'] == 'casa_rica'].copy()

In [30]:
df_test.head()

,snapshot_date,supermarket,product_id,product_name,brand,price,unit_of_measure,is_on_promotion,promotion_price,category_slug,ingestion_time,created_at
